# Reproduction: Time-Fractional Black–Scholes PDE

Reproduction of: S.M. Nuugulu, F. Gideon, K.C. Patidar, *"A robust numerical scheme for a time-fractional Black-Scholes partial differential equation describing stock exchange dynamics"*, Chaos, Solitons and Fractals 145 (2021) 110753.

See `plan.md` for the reproduction plan and resolved ambiguities decided *before* coding, and `docs/validation_report.md` for issues found *during* implementation (most importantly: the paper's printed closed-form coefficients for the `n>=1` update step are numerically unstable, and a corrected closed form was derived and validated instead).

## 1. The model

The time-fractional Black–Scholes PDE (eq. 2.13):

$$\frac{\partial^\alpha V}{\partial t^\alpha} = \left(rV - qS\frac{\partial V}{\partial S}\right)\frac{t^{1-\alpha}}{\Gamma(2-\alpha)} - \frac{\Gamma(1+\alpha)}{2}\sigma^2 S^2 \frac{\partial^2 V}{\partial S^2}, \qquad q = r-\delta,\ \ 0<\alpha\le 1$$

subject to (eq. 2.14), for a European put:

$$V(S,0)=\max(K-S,0), \qquad V(0,t)=Ke^{-r(T-t)}, \qquad \lim_{S\to\infty}V(S,t)=0.$$

The time grid $t_n = nk$ is **time-to-maturity** ($n=0$ is maturity, $n=N$ is today), so the boundary condition is implemented as $V_0^n = K e^{-r t_n}$ (plan.md 1.2–1.3). At $\alpha=1$ the fractional derivative collapses to the ordinary one and the PDE is classical Black–Scholes.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt

from src.fractional_weights import betas, phis
from src.thomas_solver import thomas
from src.classical_bs import bs_put, cn_bs_put
from src.tfbs_solver import solve_tfbs

## 2. Discretization and the corrected closed form

The Caputo derivative is discretized (eq. 3.6) via a centered difference over the fractional-order kernel, giving weights

$$\beta_j = (j+1)^{1-\alpha} - j^{1-\alpha}, \qquad \rho_\alpha = \frac{-1}{2\Gamma(2-\alpha)k^\alpha}, \qquad \frac{\partial^\alpha V(S_l,t_{n+1})}{\partial t^\alpha} \approx \rho_\alpha \sum_{j=0}^{n}\beta_j\left(V_l^{n-j+1}-V_l^{n-j-1}\right).$$

This raw sum's last term ($j=n$) always requires a fictitious pre-maturity value $V_l^{-1}$. We verified (symbolically, via `sympy`, and by matching the paper's own bootstrap step eq. 3.10 exactly) that the correct convention is the symmetric ghost point $V_l^{-1} := V_l^0$, and that the resulting per-step coefficients **must** use $\beta_0$ (which is exactly 1, for every $\alpha$) in the diagonal — not $\beta_1$, as the paper's printed eq. (3.11)-(3.12) has it. The literal printed formula was tested and found to be numerically unstable for every $\alpha$ tested (blows up to values like $10^{19}$ for a put option bounded by the strike). See `docs/validation_report.md` for the full derivation and evidence.

Let's confirm the $\beta$ weights behave as claimed (plan.md Phase 3): $\beta_0=1$ always, strictly decreasing, and collapsing to $\beta_0=1,\ \beta_{j\ge1}=0$ at $\alpha=1$.

In [ ]:
for alpha in [0.3, 0.7, 1.0]:
    b = betas(alpha, 6)
    print(f"alpha={alpha}: beta = {np.round(b, 4)}")

## 3. Stopping tests (Phases 1–6)

The full stopping-test suite lives in `tests/` and is run with `pytest tests/`. Phase 6 (the $\alpha=1$ tiebreaker) was revised from "matches a plain Crank–Nicolson solver to machine precision" to "achieves the paper's claimed double-mesh convergence rate" — see the validation report for why. We reproduce that check here.

In [ ]:
def double_mesh_error(params, alpha, N):
    L = N
    Uc = solve_tfbs(**params, L=L, N=N, alpha=alpha)
    Uf = solve_tfbs(**params, L=2*L, N=2*N, alpha=alpha)
    return np.max(np.abs(Uc[N, :] - Uf[2*N, ::2]))

params = dict(K=150.0, r=0.055, delta=0.025, sigma=0.01, T=1.0, Smax=450.0)

for alpha in [0.5, 1.0]:
    errs = [double_mesh_error(params, alpha, N) for N in (100, 200, 400)]
    rates = [np.log2(errs[i]/errs[i+1]) for i in range(len(errs)-1)]
    print(f"alpha={alpha}: errors={['%.2e' % e for e in errs]} rates={['%.2f' % r for r in rates]}")

Rates approach 2 as $N$ grows, matching the paper's claimed $O(k^2+h^2)$ scheme (Theorem 4.4) — fractional $\alpha$ reaches its asymptotic rate faster than $\alpha=1$ does, but both trend to 2.

## 4. Example 5.1: K=150, r=0.055, σ=0.01, T=1, S_max=450

Reproducing Fig. 1 (maturity payoffs at three dividend yields) and Figs. 2–4 (payoff surfaces per $\alpha$, one figure per $\delta$). Full generation script: `experiments/example_5_1.py`; figures are saved to `results/figures/`.

In [ ]:
K, R, SIGMA, T, SMAX, L, N = 150.0, 0.055, 0.01, 1.0, 450.0, 100, 100
S = np.arange(0, L+1) * (SMAX / L)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, delta in zip(axes, [0.025, 0.055, 0.065]):
    for alpha in [0.1, 0.3, 0.5, 0.7, 0.9]:
        U = solve_tfbs(K, R, delta, SIGMA, T, SMAX, L, N, alpha)
        ax.plot(S, U[N, :], label=f"alpha={alpha}")
    ax.set_title(f"delta = {delta}")
    ax.set_xlabel("S")
    ax.legend(fontsize=8)
axes[0].set_ylabel("V(S,T)")
fig.suptitle("Fig. 1: Example 5.1 maturity payoffs")
fig.tight_layout()
plt.show()

The paper notes payoff curves are smoother for $1/2 \le \alpha < 1$ than for $0<\alpha<1/2$ (§5 discussion) — the curves above are visually close together at these parameters, matching the paper's own Fig. 1 (differences between $\alpha$ values are subtle at maturity; they show up more clearly in the full payoff surfaces, Figs. 2–4, generated by `experiments/example_5_1.py`).

## 5. Example 5.2: K=200, r=0.065, σ=0.025, T=1, S_max=600

Reproducing Fig. 5 (two dividend yields) and Figs. 6–7. The paper captions **both** Fig. 6 and Fig. 7 "δ=0.085", even though Example 5.2 specifies two yields (0.045 and 0.085) and Fig. 5 correctly shows both — almost certainly a duplicated/mislabeled figure in the original publication (plan.md 1.10). `experiments/example_5_2.py` generates both δ variants explicitly rather than reproducing the apparent duplicate.

In [ ]:
K2, R2, SIGMA2, T2, SMAX2, L2, N2 = 200.0, 0.065, 0.025, 1.0, 600.0, 100, 100
S2 = np.arange(0, L2+1) * (SMAX2 / L2)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
for ax, delta in zip(axes, [0.045, 0.085]):
    for alpha in [0.1, 0.3, 0.5, 0.7, 0.9]:
        U = solve_tfbs(K2, R2, delta, SIGMA2, T2, SMAX2, L2, N2, alpha)
        ax.plot(S2, U[N2, :], label=f"alpha={alpha}")
    ax.set_title(f"delta = {delta}")
    ax.set_xlabel("S")
    ax.legend(fontsize=8)
axes[0].set_ylabel("V(S,T)")
fig.suptitle("Fig. 5: Example 5.2 maturity payoffs")
fig.tight_layout()
plt.show()

## 6. Convergence study (Tables 1–4)

`experiments/convergence.py` runs the full double-mesh study (refining $L$ and $N$ together, per plan.md Section 6) across $\alpha \in \{0.1,\ldots,1.0\}$ and $N \in \{100,200,400,800,1600\}$, for both examples, and writes `results/tables/table1_table2_example_5_1.csv` and `results/tables/table3_table4_example_5_2.csv`. It also runs the cheap stability sanity check (large time steps on a fixed fine spatial grid stay bounded and non-oscillatory — the observable content of Theorem 4.1's unconditional-stability claim).

Target shape (paper's own numbers, to compare against, not to match digit-for-digit — plan.md Phase 9): Tables 1–2 rates run from ~1.91 (N=200) up to ~1.99 (N=1600) across all $\alpha$; Tables 3–4 run from ~1.95 up to ~2.01. Load the CSVs below once `experiments/convergence.py` has finished running (it takes on the order of tens of minutes for the full $N=1600$ range, since the scheme's memory sum is inherently $O(N^2)$ per spatial point — an intrinsic cost of any fractional-derivative scheme of this type, not a performance bug).

In [ ]:
import pandas as pd

table_path = os.path.join('..', 'results', 'tables', 'table1_table2_example_5_1.csv')
if os.path.exists(table_path):
    df = pd.read_csv(table_path)
    print(df)
else:
    print('Run experiments/convergence.py first to generate this table.')

## 7. Summary and deviations from the paper

See `docs/validation_report.md` for full detail. In brief:

1. **Coefficient closed form (major deviation).** The paper's printed $n\ge1$ coefficients (using $\beta_1$ in the diagonal) are numerically unstable for every tested $\alpha$ and algebraically inconsistent with the paper's own raw discretization. A corrected closed form was derived from the raw sum, verified symbolically and via a residual check, and is what this implementation uses.
2. **$C^n$ boundary index (plan.md 1.6)** is moot under the corrected closed form, which has no such term.
3. **$\alpha=1$ vs. plain Crank–Nicolson (Phase 6).** Bit-exact matching was not achievable (the scheme's centered-difference structure differs from a standard one-step CN scheme even at $\alpha=1$); validated instead via double-mesh convergence rate, matching the paper's own Tables 1–4 methodology. A residual ~2% discrepancy against the closed-form analytic price at fixed, moderate $N$ remains unexplained and is flagged as an open item.
4. **Fig. 6/7 $\delta$ labeling (plan.md 1.10):** both $\delta$ variants are generated explicitly rather than reproducing the paper's apparent duplicate.